# MMAction2 neck supplement v5 · Colab GPU

Train one **neck assault** score from AIRTLab choke clips and hard negatives. The
original `mmaction_fall_wrestle` checkpoint remains frozen. The final alert is
`parent alert OR neck alert`, so the new head cannot turn off an existing fight
alert. The notebook measures fight recall against the parent on held-out SCFD
clips and saves all states to Drive after each epoch.

The pilot is trained from a small neck set. It is **not approved for camera use**
until additional neck examples, including CHAD clips reviewed by a human, and
real camera tests pass. This notebook creates a new ZIP; it does not install it.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, urllib.request, zipfile
from google.colab import drive, files

RUN_TAG = 'neck_supplement_v5_01'
EPOCHS = 40
TRAIN_MINUTES = 90
DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1khce9fRYpEF80osVFkOxceIdosJnwFRS'
try:
    drive.mount('/content/drive', timeout_ms=180000)
except Exception as exc:
    raise RuntimeError('Drive mount failed; use the account that opens '+DRIVE_FOLDER_URL) from exc
drive_candidates = [Path('/content/drive/MyDrive/SENTINEL'),
                    Path('/content/drive/My Drive/SENTINEL'),
                    Path('/content/drive/Shareddrives/SENTINEL'),
                    Path('/content/drive/Shared drives/SENTINEL')]
old_runs = [root/'artifacts/mmaction_release_neck_v4/release_neck_v4_02'
            for root in drive_candidates
            if (root/'artifacts/mmaction_release_neck_v4/release_neck_v4_02').is_dir()]
if len(old_runs) != 1:
    raise RuntimeError('Expected exactly one visible v4 Drive run at '+DRIVE_FOLDER_URL)
RUN = old_runs[0] / RUN_TAG
RUN.mkdir(exist_ok=True)
probe = RUN/'colab_write_probe.txt'
probe.write_text(RUN_TAG+': Drive write preflight OK', encoding='utf-8')
assert probe.read_text(encoding='utf-8').endswith('Drive write preflight OK')
subprocess.run(['nvidia-smi'], check=True)
WORK = Path('/content/neck_supplement_v5'); WORK.mkdir(exist_ok=True)
CODE = WORK/'code'; CODE.mkdir(exist_ok=True)
DEMO = CODE/'mmaction2_demo'
WEIGHTS = WORK/'weights'; WEIGHTS.mkdir(exist_ok=True)
CACHE = RUN/'feature_cache'
print('Run directory:', RUN)


In [ ]:
# Immutable project source; reruns use the same trainer and preprocessing.
SOURCE_REVISION = '9d65fa87b99c97e94c278fec919d1dc4e04e9b92'
source_zip = WORK / 'source.zip'
revision_marker = WORK / 'source_revision.txt'
if (DEMO/'train_neck_supplement_v5.py').is_file() and (
        not revision_marker.is_file() or revision_marker.read_text().strip() != SOURCE_REVISION):
    raise RuntimeError('Colab has code from another revision; restart runtime before rerunning')
if not (DEMO / 'train_neck_supplement_v5.py').is_file():
    url = 'https://codeload.github.com/TranDuon/anomaly-behavior-detector-system/zip/' + SOURCE_REVISION
    with urllib.request.urlopen(url, timeout=180) as response, source_zip.open('wb') as target:
        shutil.copyfileobj(response, target)
    with zipfile.ZipFile(source_zip) as package:
        for item in package.infolist():
            target = (WORK / item.filename).resolve()
            assert target.is_relative_to(WORK.resolve()), item.filename
        package.extractall(WORK)
    source = WORK / ('anomaly-behavior-detector-system-' + SOURCE_REVISION)
    shutil.copytree(source, CODE, dirs_exist_ok=True)
    revision_marker.write_text(SOURCE_REVISION)
manifest_path = CODE / 'benchmarks/gmdcsa24_v2_1_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['publisher_tree_sha'] == '5abac7693229900cf80f722e878fbb119211fc1c'
print('Pinned source:', SOURCE_REVISION)

In [ ]:
# Select the two small parent checkpoint files from mmaction2_demo/weights.
# This avoids Colab's Drive credential-propagation failure.
import io
PARENT_SHA = 'ddb09180284c4093189c53b7decbfcc5b51fd2d9b7700536d104524bceb5c344'
parent_pt = WEIGHTS/'interaction_fall_wrestle.pt'
parent_json = WEIGHTS/'interaction_fall_wrestle.json'

def valid_parent():
    if not parent_pt.is_file() or not parent_json.is_file():
        return False
    metadata = json.loads(parent_json.read_text(encoding='utf-8'))
    return (hashlib.sha256(parent_pt.read_bytes()).hexdigest()
            == metadata.get('checkpoint_sha256') == PARENT_SHA)

if not valid_parent():
    print('Select BOTH interaction_fall_wrestle.pt and interaction_fall_wrestle.json')
    print('from your local mmaction2_demo/weights folder.')
    print('Alternatively select fall_wrestle_parent_verified.zip if you have it.')
    uploaded = files.upload()
    if 'fall_wrestle_parent_verified.zip' in uploaded:
        with zipfile.ZipFile(io.BytesIO(uploaded['fall_wrestle_parent_verified.zip'])) as package:
            payload = package.read('interaction.pt')
            metadata = json.loads(package.read('interaction.json'))
    elif {'interaction_fall_wrestle.pt', 'interaction_fall_wrestle.json'} <= uploaded.keys():
        payload = uploaded['interaction_fall_wrestle.pt']
        metadata = json.loads(uploaded['interaction_fall_wrestle.json'])
    else:
        raise ValueError('Upload both interaction_fall_wrestle.pt and interaction_fall_wrestle.json')
    assert hashlib.sha256(payload).hexdigest() == metadata.get('checkpoint_sha256') == PARENT_SHA, 'Wrong parent checkpoint'
    parent_pt.write_bytes(payload)
    parent_json.write_text(json.dumps(metadata, ensure_ascii=False), encoding='utf-8')
assert valid_parent(), 'Parent checkpoint verification failed'
print('Verified parent:', PARENT_SHA)


In [ ]:
# Tải trên Colab. Cache ở /content; không cần tải trước trên máy Windows.
fall_zip = WORK / 'gmdcsa24_v2.1.zip'
fall_name = 'ekramalam/GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos-v2.1.zip'
fall_urls = [
    'https://zenodo.org/records/13354453/files/' + fall_name + '?download=1',
    'https://zenodo.org/api/records/13354453/files/' + fall_name + '/content',
]
expected_size = 1107545615
expected_md5 = manifest['archive_md5']
assert expected_md5 == '3d36f2c5c1a666b99639e4e9fd843efb'
def verified_fall_zip():
    if not fall_zip.is_file() or fall_zip.stat().st_size != expected_size:
        return False
    md5 = hashlib.md5()
    with fall_zip.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8*1024*1024), b''):
            md5.update(chunk)
    return md5.hexdigest() == expected_md5
if fall_zip.is_file() and fall_zip.stat().st_size >= expected_size and not verified_fall_zip():
    fall_zip.unlink()  # complete but invalid archive; start clean
for fall_url in fall_urls:
    stalled = 0
    for attempt in range(1, 41):
        if verified_fall_zip():
            break
        before = fall_zip.stat().st_size if fall_zip.exists() else 0
        result = subprocess.run(['curl', '-L', '--fail', '--connect-timeout', '30',
                                 '--speed-limit', '10240', '--speed-time', '60',
                                 '-C', '-', '-o', str(fall_zip), fall_url], check=False)
        after = fall_zip.stat().st_size if fall_zip.exists() else 0
        print(f'GMDCSA24 download {attempt}: curl={result.returncode}, {after}/{expected_size} bytes')
        if after >= expected_size and not verified_fall_zip():
            fall_zip.unlink()  # checksum failed; retry from zero
            after = 0
        stalled = stalled + 1 if after <= before else 0
        if stalled >= 3:
            break
    if verified_fall_zip():
        break
else:
    raise RuntimeError('GMDCSA24 download incomplete; rerun this cell to resume the ZIP')
print('GMDCSA24 ZIP size and MD5 verified')
FALL_EXTRACT = WORK / 'fall'; FALL_EXTRACT.mkdir(exist_ok=True)
with zipfile.ZipFile(fall_zip) as package:
    assert sum(item.file_size for item in package.infolist()) < 5_000_000_000
    for item in package.infolist():
        assert (FALL_EXTRACT / item.filename).resolve().is_relative_to(FALL_EXTRACT.resolve()), item.filename
        assert ((item.external_attr >> 16) & 0o170000) != 0o120000, 'Symlink in archive'
    package.extractall(FALL_EXTRACT)
first = list(FALL_EXTRACT.rglob('Subject 1/Fall/01.mp4'))
assert len(first) == 1
FALL_ROOT = first[0].parents[2]
assert (FALL_ROOT / 'Subject 4/ADL/01.mp4').is_file()

SCFD_REVISION = 'ff83b7c521e5ca4eb67a212cc42dbc54f39d6290'
SCFD_ROOT = WORK / SCFD_REVISION
if not SCFD_ROOT.is_dir():
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/seymanurakti/fight-detection-surv-dataset.git', str(SCFD_ROOT)], check=True)
    subprocess.run(['git', '-C', str(SCFD_ROOT), 'checkout', '--quiet', SCFD_REVISION], check=True)
assert subprocess.check_output(['git','-C',str(SCFD_ROOT),'rev-parse','HEAD'],text=True).strip() == SCFD_REVISION
assert len(list((SCFD_ROOT/'fight').glob('*.mp4'))) == 150
assert len(list((SCFD_ROOT/'noFight').glob('*.mp4'))) == 150
print('Data ready on Colab:', FALL_ROOT, SCFD_ROOT)


In [ ]:
# Môi trường riêng tương thích MMAction2 v1.2.0 và checkpoint v1.
os.environ.update(MPLBACKEND='Agg', CUDA_MODULE_LOADING='LAZY', OMP_NUM_THREADS='4')
ENV = WORK / 'env'; PY = ENV / 'bin/python'
def run(command, label, timeout=1800):
    log = RUN / (label + '.log')
    with log.open('a', encoding='utf-8') as output:
        result = subprocess.run([str(x) for x in command], stdout=output, stderr=subprocess.STDOUT, timeout=timeout)
    if result.returncode:
        raise RuntimeError(f'{label} failed; see {log}\n{log.read_text(encoding="utf-8")[-4000:]}')
    print(label, 'OK')
if not PY.is_file():
    run([sys.executable,'-m','pip','install','-q','uv==0.8.22'], 'install_uv')
    run(['uv','python','install','3.10'], 'install_python')
    run(['uv','venv','--python','3.10','--seed',ENV], 'create_env')
run([PY,'-m','pip','install','-q','numpy==1.26.4','setuptools==69.5.1','wheel'], 'install_base')
run([PY,'-m','pip','install','-q','torch==2.0.1+cu118','torchvision==0.15.2+cu118',
     '--index-url','https://download.pytorch.org/whl/cu118'], 'install_torch')
run([PY,'-m','pip','install','-q','mmcv==2.1.0','--only-binary=mmcv',
     '-f','https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html',
     'mmengine==0.10.7','mmdet==3.2.0','decord==0.6.0','einops==0.8.1',
     'importlib-metadata','scikit-learn==1.3.2','opencv-python==4.8.1.78'], 'install_openmmlab')
if not (DEMO / 'repo/.git').is_dir():
    run(['git','clone','--depth','1','--branch','v1.2.0','https://github.com/open-mmlab/mmaction2.git',DEMO/'repo'], 'clone_mmaction')
run([PY,'-m','pip','install','--no-deps','--no-build-isolation','-e',DEMO/'repo'], 'install_mmaction')
os.environ['PYTHONPATH'] = os.pathsep.join(map(str,(CODE,DEMO,DEMO/'repo')))
run([PY,'-c','import torch, mmcv; from mmcv.ops import roi_align; assert torch.cuda.is_available()'], 'check_cuda')


In [ ]:
# Public AVA and person detector weights go directly into /content.
checks = {
 'action_slowonly_r101.pth': (
  'https://download.openmmlab.com/mmaction/detection/ava/slowonly_omnisource_pretrained_r101_8x8x1_20e_ava_rgb/slowonly_omnisource_pretrained_r101_8x8x1_20e_ava_rgb_20201217-16378594.pth',
  '163785949c5178484012a6593d12d6fcf4d36be404fa42b5899b314e91f4703b'),
 'person_faster_rcnn_r50.pth': (
  'https://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth',
  'a5d8aa15c75b961b9c73f28a6e92af370835639a2939ff890852b052a5256a5b')}
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
for name,(url,expected) in checks.items():
    target = WEIGHTS/name
    if not target.is_file() or sha256(target)!=expected:
        target.unlink(missing_ok=True)
        for attempt in range(1,4):
            temporary=target.with_suffix('.partial')
            temporary.unlink(missing_ok=True)
            try:
                subprocess.run(['curl','-L','--fail','--retry','3','-o',str(temporary),url],check=True)
                if sha256(temporary)==expected:
                    temporary.replace(target)
                    break
            except Exception as exc:
                print(name, 'attempt', attempt, exc)
            temporary.unlink(missing_ok=True)
        if not target.is_file():
            raise RuntimeError(name+' public download failed; rerun this cell')
        assert sha256(target)==expected, name+' checksum mismatch'
print('AVA and detector weights verified')

In [ ]:
# Re-DID fight/after-fight transitions; source archives stay in /content.
review_root = WORK/'review'; review_root.mkdir(exist_ok=True)
review_manifest = review_root/'review.json'
redid_archives = [
 ('ReDIDvideos.zip', 'https://loki.disi.unitn.it/~ReDID/REDID_small/fights/ReDIDvideos.zip', 'b8874d3cc07b8310398e27438d64f399505d2a85daec229d9c8d2cd91773b346'),
 ('actions_bboxes.zip', 'https://loki.disi.unitn.it/~ReDID/REDID_small/fights/actions_bboxes.zip', '0b7571357b01c29def9009b3335b73c7251aa818fc24eb15c910d2f40b8d062b'),
]
for name,url,expected in redid_archives:
    path=WORK/name
    if not path.is_file() or sha256(path)!=expected:
        path.unlink(missing_ok=True)
        subprocess.run(['curl','-L','--fail','--retry','5','--retry-delay','5',
                        '-C','-','-o',str(path),url],check=True)
    assert sha256(path)==expected, name+' checksum mismatch'
assert shutil.which('ffmpeg')
run([PY,CODE/'scripts/build_redid_transitions.py',
     '--video-archive',WORK/'ReDIDvideos.zip','--action-archive',WORK/'actions_bboxes.zip',
     '--review-root',review_root], 'build_redid_transitions', timeout=3600)
for name,_,_ in redid_archives:
    (WORK/name).unlink()

# AIRTLab: clip-level choke positives and hug negatives, both camera views.
AIRTLAB_REVISION='1f7747e104301ccaa82ef5a2f6804b51ced1c398'
AIRTLAB_ROOT=WORK/'airtlab'
samples={'violent':[26,33,37,38,80,81,83,87,89,98,106],
         'non-violent':[1,5,6,12,25,28,29,32,33,36,37,38,44,47,49,58]}
for category,ids in samples.items():
    for camera in ('cam1','cam2'):
        folder=AIRTLAB_ROOT/category/camera; folder.mkdir(parents=True,exist_ok=True)
        for number in ids:
            target=folder/f'{number}.mp4'
            if target.is_file() and target.stat().st_size>10000:
                continue
            url=('https://raw.githubusercontent.com/airtlab/A-Dataset-for-Automatic-Violence-Detection-in-Videos/'
                 +AIRTLAB_REVISION+'/violence-detection-dataset/'+category+'/'+camera+'/'+target.name)
            temporary=target.with_suffix('.partial')
            subprocess.run(['curl','-L','--fail','--retry','5','--retry-delay','3',
                            '-o',str(temporary),url],check=True)
            assert temporary.stat().st_size>10000, url
            temporary.replace(target)
print('Re-DID and AIRTLab ready in /content')

In [ ]:
base = [PY, DEMO/'train_neck_supplement_v5.py', '--manifest', manifest_path,
        '--fall-root', FALL_ROOT, '--scfd-root', SCFD_ROOT,
        '--airtlab-root', AIRTLAB_ROOT,
        '--review-manifest', review_manifest, '--review-root', review_root,
        '--weights', WEIGHTS, '--cache', CACHE, '--output', RUN,
        '--device', 'cuda:0', '--epochs', EPOCHS, '--minutes', TRAIN_MINUTES]
run(base + ['--stage', 'features'], 'extract_features', timeout=8*3600)
run(base + ['--stage', 'train'], 'train_neck_supplement', timeout=(TRAIN_MINUTES+10)*60)


In [ ]:
report_path = RUN/'report.json'
assert report_path.is_file(), 'Training did not finish; inspect logs in '+str(RUN)
result = json.loads(report_path.read_text(encoding='utf-8'))
assert result['status'] == 'completed'
assert result['test']['fight_recall_nonregression']
package = RUN/'mmaction_neck_supplement_v5_pilot.zip'
assert package.is_file() and package.stat().st_size > 1000000
with zipfile.ZipFile(package) as saved:
    assert {'neck_supplement.pt','neck_supplement.json','report.json','protocol.json'} == set(saved.namelist())
    assert saved.testzip() is None, 'Pilot ZIP is damaged'
print('Held-out test:', json.dumps(result['test'], indent=2))
print('Pilot ZIP SHA-256:', sha256(package))
print('Flushing files to Google Drive...')
drive.flush_and_unmount(timeout_ms=600000)
DRIVE_FLUSHED = True
print('Saved and flushed:', DRIVE_FOLDER_URL, RUN_TAG)


In [ ]:
# Disconnect only after the verified ZIP and report have been flushed to Drive.
assert DRIVE_FLUSHED, 'Drive output was not verified and flushed'
from google.colab import runtime
runtime.unassign()
